In [22]:
import os

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import WebBaseLoader, PDFMinerLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_classic import hub

In [19]:
from dotenv import load_dotenv

load_dotenv()

True

In [20]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.environ['CLOSEAI_API_KEY'],
    base_url=os.environ['CLOSEAI_BASE_URL']
)
# embedding model 
embedding_model = OpenAIEmbeddings(
    api_key=os.environ['CLOSEAI_API_KEY'],
    base_url=os.environ['CLOSEAI_BASE_URL']
)

In [24]:
if not os.path.exists('local_save'):
    # 加载网页中文本内容，转换langchain处理的Document
    loader = PDFMinerLoader('./data/The Era of Experience Paper.pdf')
    docs = loader.load()

    # TextSplitter实现加载后Document分割
    splitter = RecursiveCharacterTextSplitter(
        separators=['\n\n','\n',''],
        chunk_size=1000,
        chunk_overlap=100
    )
    splited_docs = splitter.split_documents(docs)
    
    # 创建向量数据库（内存中）对chunk进行向量化和存储
    vector_store = FAISS.from_documents(
        documents=splited_docs,
        embedding=embedding_model)
    # 向量数据库本地化存储
    vector_store.save_local('local_save')
    print('faiss数据库本地化保存成功！')
else:
    vector_store = FAISS.load_local(
        'local_save', 
        embeddings=embedding_model, 
        allow_dangerous_deserialization=True
    )
    print('加载faiss数据库本地化记录成功！')



faiss数据库本地化保存成功！


In [25]:
# 构建检索器
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

def format_docs(docs):
    return '\n\n'.join([doc.page_content for doc in docs])

# prompt 
prompt = hub.pull('rlm/rag-prompt')

# 构建rag chain
rag_chain = (
    {"context": retriever | format_docs , "question": RunnablePassthrough()} 
    | prompt 
    | llm 
    | StrOutputParser()
)

e:\miniconda3\envs\py3127\Lib\site-packages\langsmith\client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [27]:
# rag检索
response = rag_chain.invoke("介绍一下paper的整体内容, 以中文输出")
print(response)

这篇论文讨论了人类中心人工智能（AI）向更先进的模式的过渡，特别是在数学能力方面的进展。举例来说，AlphaProof 首次在国际数学奥林匹克中获奖，超越了以人为中心的方法。整体而言，文章探讨了大型语言模型在推理和证明方面的能力和影响。
